<a href="https://colab.research.google.com/github/TVTruong777/DL261-Assignment/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import pandas as pd

In [ ]:
go_tion = load_dataset("google-research-datasets/go_emotions", "simplified")

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
go_labels = go_tion["train"].features["labels"].feature.names

In [ ]:
go_train = go_tion["train"].to_pandas()
go_val = go_tion["validation"].to_pandas()
go_test = go_tion["test"].to_pandas()

In [ ]:
go_train["emotion"] = go_train["labels"].apply(lambda ids: [go_labels[i] for i in ids])
go_val["emotion"] = go_val["labels"].apply(lambda ids: [go_labels[i] for i in ids])
go_test["emotion"] = go_test["labels"].apply(lambda ids: [go_labels[i] for i in ids])

In [ ]:
print(f"overlapping ids between train and validation set: {len(set(go_train["id"]) & set(go_val["id"]))}")
print(f"overlapping ids between train and test set: {len(set(go_train["id"]) & set(go_test["id"]))}")

go_train = go_train.drop(columns = ["id"])
go_val = go_val.drop(columns = ["id"])
go_test = go_test.drop(columns = ["id"])

overlapping ids between train and validation set: 0
overlapping ids between train and test set: 0


In [ ]:
df = [("train", go_train), ("validation", go_val), ("test", go_test)]
for go_set in df:
  duplicates = go_set[1]["text"].duplicated(keep = False)
  print(f"there are {duplicates.sum()} duplicates in {go_set[0]} set")
  print(go_set[1][duplicates].sort_values(by = "text").head(10))

there are 301 duplicates in train set
                                            text  labels  \
1030   A surprise to be sure, but a welcome one.    [20]   
41866  A surprise to be sure, but a welcome one.    [13]   
22935  A surprise to be sure, but a welcome one.    [26]   
16095                              Amen to that!     [4]   
37409                              Amen to that!    [27]   
26898                                 Anytime :)    [27]   
891                                   Anytime :)    [27]   
27042                           Are you serious?  [6, 7]   
30285                           Are you serious?     [7]   
8434                            Are you serious?     [7]   

                      emotion  
1030               [optimism]  
41866            [excitement]  
22935              [surprise]  
16095              [approval]  
37409               [neutral]  
26898               [neutral]  
891                 [neutral]  
27042  [confusion, curiosity]  
30285        

In [ ]:
def group_labels(label_list):
  unique_labels = sorted(set(label for labels in label_list for label in labels))
  if len(unique_labels) > 1 and 27 in unique_labels:
    unique_labels.remove(27)
  return unique_labels

In [ ]:
go_group = go_train.groupby("text", as_index = False).agg({"labels": group_labels})
go_group["emotion"] = go_group["labels"].apply(lambda emo: [go_labels[e] for e in emo])
go_group[go_group["labels"].apply(lambda labels: 27 in labels)]

,text,labels,emotion
3,"'*Pray*, v. To ask that the laws of the unive...",[27],[neutral]
4,">it'll get invaded by tankie, unfortunately. ...",[27],[neutral]
7,"Change is hard. Find comfort in victory, even...",[27],[neutral]
10,I think this discussion would benefit from di...,[27],[neutral]
13,"Luckily from him, there is no death penalty i...",[27],[neutral]
...,...,...,...
43193,🎶Hello darkness my old friend...,[27],[neutral]
43194,🎶Living in America🎶,[27],[neutral]
43195,👻 Windows 10 checking in.,[27],[neutral]
43206,😂😂😂 YES and she fired all her bridesmaids beca...,[27],[neutral]


In [ ]:
pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 14.8 MB/s  0:00:00


In [ ]:
import emoji
has_emoji = go_group["text"].apply(lambda emo: emoji.emoji_count(emo) > 0)
print(f"number of text examples that has emoji: {has_emoji.sum()}")
display(go_group[has_emoji][["text", "emotion"]])

number of text examples that has emoji: 841


,text,emotion
11,"I'm scared to even ask my mom ,I might get ye...",[fear]
92,"""I awoke several hours later in a daze...."" 😂",[realization]
273,"#FURRIES 😾 OUT, OUT! ^👉 #🤚 WE NEED A WALL 🖐️ A...",[neutral]
364,*Bud Light presents...real men of genius. 🎶 Mr...,[admiration]
415,*Questions 3 ❤️,[neutral]
...,...,...
43222,🤣🤣🤣🤣 I’m sorry for your misfortune but it’s st...,"[amusement, sadness]"
43223,🤷‍♂️ weird flex but ok,"[approval, disgust]"
43224,🤷🏻‍♀️ As a wise man once said: he was a bastar...,"[anger, annoyance]"
43225,🤷🏼‍♀️ I was wondering the same thing. It looke...,[surprise]


In [ ]:
def normalize_emoji(text):
  return emoji.demojize(text, delimiters = [" ", " "])

go_group["text"] = go_group["text"].apply(normalize_emoji)
go_val["text"] = go_val["text"].apply(normalize_emoji)
go_test["text"] = go_test["text"].apply(normalize_emoji)

display(go_group[has_emoji][["text", "emotion"]])

,text,emotion
11,"I'm scared to even ask my mom ,I might get ye...",[fear]
92,"""I awoke several hours later in a daze...."" f...",[realization]
273,"#FURRIES pouting_cat OUT, OUT! ^ backhand_in...",[neutral]
364,*Bud Light presents...real men of genius. mus...,[admiration]
415,*Questions 3 red_heart,[neutral]
...,...,...
43222,rolling_on_the_floor_laughing rolling_on_the...,"[amusement, sadness]"
43223,man_shrugging weird flex but ok,"[approval, disgust]"
43224,woman_shrugging_light_skin_tone As a wise ma...,"[anger, annoyance]"
43225,woman_shrugging_medium-light_skin_tone I was...,[surprise]


In [ ]:
import numpy as np

class_count = np.zeros(len(go_labels), dtype = int)

for idx, lb in enumerate(go_labels):
  class_count[idx] = go_group["emotion"].apply(lambda label_list: lb in label_list).sum()

class_df = pd.DataFrame({"Emotion": go_labels,
                         "Count": class_count,
                         "Class distribution": class_count * 100 / np.sum(class_count),
                         "Comment distribution": class_count * 100 / len(go_group)}).sort_values(by = "Count", ascending = False).reset_index(drop = True)
display(class_df)

,Emotion,Count,Class distribution,Comment distribution
0,neutral,12776,25.781455,29.555602
1,admiration,4120,8.313995,9.531080
2,approval,2939,5.930784,6.798991
3,gratitude,2622,5.291091,6.065653
4,annoyance,2469,4.982343,5.711708
5,amusement,2327,4.695793,5.383210
6,curiosity,2187,4.413278,5.059338
7,love,2068,4.173141,4.784047
8,disapproval,2020,4.076279,4.673005
9,optimism,1578,3.184341,3.650496


In [ ]:
temp_df = pd.DataFrame({"Number of Labels": go_group["labels"].apply(len)})
count_df = (temp_df.groupby(by = "Number of Labels", as_index = False).size().rename(columns = {"size": "Count"}))
count_df["Percentage"] = count_df["Count"] * 100 / count_df["Count"].sum()
display(count_df)

,Number of Labels,Count,Percentage
0,1,37439,86.610220
1,2,5278,12.209961
2,3,481,1.112730
3,4,28,0.064774
4,5,1,0.002313


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")
token_length = go_group["text"].apply(lambda txt: len(tokenizer.encode(txt)))
print(token_length.describe(percentiles = [0.5, 0.9, 0.95, 0.99]))

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

count    43227.000000
mean        20.360053
std          9.382955
min          3.000000
50%         20.000000
90%         33.000000
95%         36.000000
99%         42.000000
max        316.000000
Name: text, dtype: float64


In [ ]:
display(go_group.loc[token_length.nlargest(10).index, ["text", "emotion"]])
display(token_length.nlargest(10))

,text,emotion
41024,here you go: |Games|Home|Away|Team|vs W-L 18-1...,[neutral]
43207,face_with_tears_of_joy face_with_tears_of_jo...,[approval]
36297,Wow that's so lit fire fire fire face_with_...,[admiration]
19630,LOL get it guys the joke is necrophilia face_...,[amusement]
40439,"ackchyually, it's *r/woooosh ^^^^^I'm ^^^^^a ^...",[neutral]
1185,">It also means no applause lines. No problem, ...",[neutral]
273,"#FURRIES pouting_cat OUT, OUT! ^ backhand_in...",[neutral]
23457,Oh fuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu...,[anger]
42596,u/dances-with-corgis Says they'll never forget...,[amusement]
43205,face_with_tears_of_joy face_with_tears_of_jo...,[approval]


,text
41024,316
43207,159
36297,151
19630,124
40439,120
1185,99
273,95
23457,86
42596,85
43205,83


In [ ]:
emo = go_group[has_emoji]["text"].apply(lambda txt: len(tokenizer.encode(txt)) < 8)
display(go_group[has_emoji][emo].head(10))
display(go_group[has_emoji][emo]["text"].iloc[:10].apply(lambda txt: tokenizer.encode(txt)))

,text,labels,emotion
2010,All good thumbs_up,[4],[approval]
5176,Damn perfect OK_hand,[0],[admiration]
8422,Happy Birthday! sparkles,"[0, 13]","[admiration, excitement]"
13248,I love this! Australia,[18],[love]
20998,Me too two_hearts,[4],[approval]
26649,So beautiful two_hearts,[0],[admiration]
28254,Thank you smiling_face,[15],[gratitude]
28255,Thank you thumbs_up,[15],[gratitude]
28259,Thank you folded_hands,[15],[gratitude]


,text
2010,"[101, 1398, 1363, 18680, 168, 1146, 102]"
5176,"[101, 8441, 3264, 10899, 168, 1289, 102]"
8422,"[101, 8325, 16168, 106, 14798, 2897, 102]"
13248,"[101, 146, 1567, 1142, 106, 1754, 102]"
20998,"[101, 2508, 1315, 1160, 168, 12261, 102]"
26649,"[101, 1573, 2712, 1160, 168, 12261, 102]"
28254,"[101, 4514, 1128, 5278, 168, 1339, 102]"
28255,"[101, 4514, 1128, 18680, 168, 1146, 102]"
28259,"[101, 4514, 1128, 6443, 168, 1493, 102]"


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes = list(range(len(go_labels))))
y_train = mlb.fit_transform(go_group["labels"])
y_val = mlb.transform(go_val["labels"])
y_test = mlb.transform(go_test["labels"])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer(max_features = 20000, ngram_range = (1, 2))
x_train = tf_idf.fit_transform(go_group["text"])
x_val = tf_idf.transform(go_val["text"])
x_test = tf_idf.transform(go_test["text"])

print(f"Training: {x_train.shape} \nValidation: {x_val.shape} \nTest: {x_test.shape}")

Training: (43227, 20000) 
Validation: (5426, 20000) 
Test: (5427, 20000)


In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
import time

start = time.time()
multi_clf = OneVsRestClassifier(LogisticRegression(max_iter = 1000, C = 2.0), n_jobs = -1)
multi_clf.fit(x_train, y_train)
end = time.time()

W = np.vstack([clf.coef_ for clf in multi_clf.estimators_])
print(f"Training time: {end - start:.4f} seconds, Weights matrix shape: {W.shape}")

Training time: 2.5542 seconds, Weights matrix shape: (28, 20000)


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

start = time.time()
prob = multi_clf.predict_proba(x_val)
end = time.time()
idf_inf_time = end - start
print(f"Inference time: {idf_inf_time:.4f} seconds")
for threshold in [0.7, 0.5, 0.3, 0.1]:
  prediction = (prob > threshold).astype(int)
  accuracy = accuracy_score(y_true = y_val, y_pred = prediction)
  macro_f1 = f1_score(y_true = y_val, y_pred = prediction, average = "macro", zero_division = 0)
  micro_f1 = f1_score(y_true = y_val, y_pred = prediction, average = "micro", zero_division = 0)
  print(f"Threshold: {threshold} --> Accuracy (Exact match): {accuracy:.4f}, Macro F1: {macro_f1:.4f}, Micro F1: {micro_f1:.4f}")


Inference time: 0.0192 seconds
Threshold: 0.7 --> Accuracy (Exact match): 0.1474, Macro F1: 0.1849, Micro F1: 0.2597
Threshold: 0.5 --> Accuracy (Exact match): 0.2735, Macro F1: 0.2652, Micro F1: 0.4131
Threshold: 0.3 --> Accuracy (Exact match): 0.3951, Macro F1: 0.3517, Micro F1: 0.5256
Threshold: 0.1 --> Accuracy (Exact match): 0.2193, Macro F1: 0.4368, Micro F1: 0.5134


In [ ]:
feature_names = np.array(tf_idf.get_feature_names_out())
for emo_name in ["gratitude", "neutral", "amusement", "grief"]:
  idx = go_labels.index(emo_name)
  top_emos = feature_names[np.argsort(W[idx])[-6:][::-1]]
  print(f"Emotion: {emo_name} --> Top indicator: {top_emos}")

Emotion: gratitude --> Top indicator: ['thanks' 'thank' 'thank you' 'congrats' 'welcome' 'thanks for']
Emotion: neutral --> Top indicator: ['no thanks' 'they' 'you don' 'hello' 'more like' 'political']
Emotion: amusement --> Top indicator: ['lol' 'haha' 'funny' 'lmao' 'hilarious' 'fun']
Emotion: grief --> Top indicator: ['died' 'rip' 'loss' 'death' 'sorry' 'sorry for']


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

db_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")
train_ds = Dataset.from_dict({"text": go_group["text"].tolist(), "labels": y_train.astype(np.float32).tolist()})
val_ds = Dataset.from_dict({"text": go_val["text"].tolist(), "labels": y_val.astype(np.float32).tolist()})
test_ds = Dataset.from_dict({"text": go_test["text"].tolist(), "labels": y_test.astype(np.float32).tolist()})

def tokenize_batch(batch):
  return db_tokenizer(batch["text"], truncation = True, max_length = 64)

train_ds = train_ds.map(tokenize_batch, batched = True, remove_columns = ["text"])
val_ds = val_ds.map(tokenize_batch, batched = True, remove_columns = ["text"])
test_ds = test_ds.map(tokenize_batch, batched = True, remove_columns = ["text"])

Map:   0%|          | 0/43227 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
import torch

In [ ]:
def bert_train_prep(MODEL_NAME):
  if MODEL_NAME != "distilbert-base-cased" and MODEL_NAME != "bert-base-cased":
    raise ValueError(f"Passed in model name <{MODEL_NAME}> -> Model name must be <distilbert-base-cased> or <bert-base-cased>")

  model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = len(go_labels), problem_type = "multi_label_classification")

  total_parameters = sum(p.numel() for p in model.parameters())
  print(f"{MODEL_NAME} pre-trained total params: {total_parameters}")

  def compute_metrics(eval_batch):
    logits, labels = eval_batch
    prob = 1 / (1 + np.exp(-logits))
    pred = (prob >= 0.3).astype(int)
    return {
        "accuracy": accuracy_score(y_true = labels, y_pred = pred),
        "macro-f1": f1_score(y_true = labels, y_pred = pred, average = "macro", zero_division = 0),
        "micro-f1": f1_score(y_true = labels, y_pred = pred, average = "micro", zero_division = 0)
    }

  train_args = TrainingArguments(output_dir = f"./{MODEL_NAME}_goemotion", save_strategy = "epoch", eval_strategy = "epoch",
                                per_device_train_batch_size = 32, per_device_eval_batch_size = 64, num_train_epochs = 4,
                                learning_rate = 3e-5, weight_decay = 0.01, warmup_steps = 0.1, save_total_limit = 1,
                                metric_for_best_model = "macro-f1", load_best_model_at_end = True, greater_is_better = True,
                                fp16 = torch.cuda.is_available(), report_to = "none")

  trainer = Trainer(model = model, args = train_args, train_dataset = train_ds, eval_dataset = val_ds,
                    compute_metrics = compute_metrics, data_collator = DataCollatorWithPadding(tokenizer = db_tokenizer))

  return trainer

In [ ]:
dbert = bert_train_prep("distilbert-base-cased")
dbert_result = dbert.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distilbert-base-cased pre-trained total params: 65803036


Epoch,Training Loss,Validation Loss,Accuracy,Macro-f1,Micro-f1
1,0.114566,0.092124,0.453741,0.368114,0.572060
2,0.084758,0.085426,0.465168,0.439259,0.597979
3,0.072596,0.085660,0.464615,0.470189,0.600203
4,0.063636,0.087706,0.457427,0.481749,0.589155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
bert = bert_train_prep("bert-base-cased")
bert_result = bert.train()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert-base-cased pre-trained total params: 108331804


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [ ]:
def sweep_threshold(bert_model, name, val_ds, threshold = (0.1, 0.2, 0.3, 0.4, 0.5)):
  if torch.cuda.is_available():
    torch.cuda.synchronize()
  start = time.perf_counter()
  pred = bert_model.predict(val_ds)
  if torch.cuda.is_available():
    torch.cuda.synchronize()
  end = time.perf_counter()
  inf_time = end - start
  probs = 1 / (1 + np.exp(-pred.predictions))
  val_labels = pred.label_ids
  for t in threshold:
    prediction = (probs >= t).astype(int)
    accuracy = accuracy_score(y_true = val_labels, y_pred = prediction)
    macro_f1 = f1_score(y_true = val_labels, y_pred = prediction, average = "macro", zero_division = 0)
    micro_f1 = f1_score(y_true = val_labels, y_pred = prediction, average = "micro", zero_division = 0)
    print(f"Model: {name}, Threshold {t} --> accuracy: {accuracy:.4f}, macro_f1: {macro_f1:.4f}, micro_f1: {micro_f1:.4f}")
  print(f"========================= {name} inference time {inf_time:.4f} =========================")
  return inf_time

In [ ]:
dbert_inf_time = sweep_threshold(bert_model = dbert, name = "Distil-bert", val_ds = val_ds)
bert_inf_time = sweep_threshold(bert_model = bert, name = "Bert", val_ds = val_ds)

Model: Distil-bert, Threshold 0.1 --> accuracy: 0.2921, macro_f1: 0.4788, micro_f1: 0.5498
Model: Distil-bert, Threshold 0.2 --> accuracy: 0.3986, macro_f1: 0.4913, micro_f1: 0.5846
Model: Distil-bert, Threshold 0.3 --> accuracy: 0.4574, macro_f1: 0.4817, micro_f1: 0.5892
Model: Distil-bert, Threshold 0.4 --> accuracy: 0.4727, macro_f1: 0.4630, micro_f1: 0.5854
Model: Distil-bert, Threshold 0.5 --> accuracy: 0.4513, macro_f1: 0.4277, micro_f1: 0.5684
========================= Distil-bert inference time 1.9160 =========================


Model: Bert, Threshold 0.1 --> accuracy: 0.2997, macro_f1: 0.4487, micro_f1: 0.5463
Model: Bert, Threshold 0.2 --> accuracy: 0.4115, macro_f1: 0.4758, micro_f1: 0.5883
Model: Bert, Threshold 0.3 --> accuracy: 0.4633, macro_f1: 0.4675, micro_f1: 0.5926
Model: Bert, Threshold 0.4 --> accuracy: 0.4711, macro_f1: 0.4523, micro_f1: 0.5837
Model: Bert, Threshold 0.5 --> accuracy: 0.4539, macro_f1: 0.4272, micro_f1: 0.5699
========================= Bert inference time 3.3920 =========================


In [ ]:
dbert_params = dbert.model.num_parameters()
bert_params = bert.model.num_parameters()

dbert_metrics = dbert.evaluate()
bert_metrics = bert.evaluate()
dbert_metrics["model"] = "Distil - Bert"
bert_metrics["model"] = "Bert"
dbert_metrics["Inference time"] = dbert_inf_time
bert_metrics["Inference time"] = bert_inf_time
dbert_metrics["Total parameters"] = dbert_params
bert_metrics["Total parameters"] = bert_params

idf_prediction = (prob > 0.3).astype(int)
idf_accuracy = accuracy_score(y_true = y_val, y_pred = idf_prediction)
idf_macro_f1 = f1_score(y_true = y_val, y_pred = idf_prediction, average = "macro", zero_division = 0)
idf_micro_f1 = f1_score(y_true = y_val, y_pred = idf_prediction, average = "micro", zero_division = 0)
idf_params = sum(clf.coef_.size + clf.intercept_.size for clf in multi_clf.estimators_)
idf_metrics = {"model": "TF-IDF", "eval_accuracy": idf_accuracy, "eval_macro-f1": idf_macro_f1, "eval_micro-f1": idf_micro_f1, "Inference time": idf_inf_time, "Total parameters": idf_params}


metrics = []
metrics.append(idf_metrics)
metrics.append(dbert_metrics)
metrics.append(bert_metrics)

Training Loss,Validation Loss,Epoch,Accuracy,Macro-f1,Micro-f1
0.063636,0.087706,4,0.457427,0.481749,0.589155


Training Loss,Validation Loss,Epoch,Accuracy,Macro-f1,Micro-f1
0.063570,0.088828,4,0.463325,0.467484,0.592610


In [ ]:
model_metrics = pd.DataFrame(metrics).drop(columns = ["eval_loss"])
model_metrics.columns = ["Model", "Accuracy", "Macro-f1", "Micro-f1", "Inference time", "Total parameters"]
display(model_metrics)

,Model,Accuracy,Macro-f1,Micro-f1,Inference time,Total parameters
0,TF-IDF,0.395135,0.351750,0.525628,0.019158,560028
1,Distil - Bert,0.457427,0.481749,0.589155,1.916049,65803036
2,Bert,0.463325,0.467484,0.592610,3.391975,108331804


In [ ]:
from sklearn.metrics import classification_report

final_threshold = 0.3
final_model = dbert

start = time.perf_counter()
pred = final_model.predict(test_ds)
end = time.perf_counter()
test_inf_time = end - start
probs = 1 / (1 + np.exp(-pred.predictions))
prediction = (probs >= final_threshold).astype(int)
test_labels = pred.label_ids
accuracy = accuracy_score(y_true = test_labels, y_pred = prediction)
macro_f1 = f1_score(y_true = test_labels, y_pred = prediction, average = "macro", zero_division = 0)
micro_f1 = f1_score(y_true = test_labels, y_pred = prediction, average = "micro", zero_division = 0)
print(f"Distil-bert test result --> Inference time: {test_inf_time:.4f}, Accuracy: {accuracy:.4f}, Macro-f1: {macro_f1:.4f}, Micro-f1: {micro_f1:.4f}")
print(classification_report(test_labels, prediction, target_names = go_labels, zero_division = 0))

Distil-bert test result --> Inference time: 1.9455, Accuracy: 0.4520, Macro-f1: 0.4777, Micro-f1: 0.5940
                precision    recall  f1-score   support

    admiration       0.62      0.74      0.68       504
     amusement       0.77      0.91      0.83       264
         anger       0.44      0.53      0.48       198
     annoyance       0.41      0.37      0.39       320
      approval       0.43      0.39      0.41       351
        caring       0.42      0.40      0.41       135
     confusion       0.44      0.46      0.45       153
     curiosity       0.48      0.64      0.55       284
        desire       0.56      0.42      0.48        83
disappointment       0.36      0.26      0.31       151
   disapproval       0.38      0.40      0.39       267
       disgust       0.55      0.51      0.53       123
 embarrassment       0.87      0.35      0.50        37
    excitement       0.49      0.46      0.47       103
          fear       0.61      0.76      0.67        7